In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
from scipy import stats


In [2]:
tracks = pd.read_csv("project4/Data/music_tracks.csv")
artists = pd.read_csv("project4/Data/artists.csv")

## 1. Introduction

This project uses the Spotify Music Tracks dataset. Each row represents one Spotify track and includes metadata such as the track name, artist name, popularity score, duration, release date, explicit status, and genre. The dataset also includes audio features such as danceability, energy, valence, acousticness, instrumentalness, loudness, tempo, and speechiness.

The main question I want to explore is:

**Can audio features and genre help predict whether a Spotify track is popular?**

In [3]:
tracks.head()

,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,release_date,explicit,danceability,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,1974,False,0.676,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,1995-04,False,0.420,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,1973,False,0.438,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,2018-08-10,False,0.266,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,2017-02-03,False,0.618,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,NaN,4,acoustic


In [4]:
tracks.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114000 entries, 0 to 113999
Data columns (total 22 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Unnamed: 0        114000 non-null  int64  
 1   track_id          114000 non-null  object 
 2   artists           113999 non-null  object 
 3   album_name        113999 non-null  object 
 4   track_name        113999 non-null  object 
 5   popularity        114000 non-null  int64  
 6   duration_ms       114000 non-null  int64  
 7   release_date      114000 non-null  object 
 8   explicit          114000 non-null  bool   
 9   danceability      114000 non-null  float64
 10  energy            114000 non-null  float64
 11  key               114000 non-null  int64  
 12  loudness          114000 non-null  float64
 13  mode              114000 non-null  int64  
 14  speechiness       114000 non-null  float64
 15  acousticness      114000 non-null  float64
 16  instrumentalness  11

In [5]:
print("Tracks dataset shape:", tracks.shape)
print("Artists dataset shape:", artists.shape)

Tracks dataset shape: (114000, 22)
Artists dataset shape: (1162095, 5)


The Spotify project uses two datasets. The main dataset, `music_tracks.csv`, contains 114,000 rows and 22 columns. Each row represents one Spotify track with audio features, metadata, popularity, and genre information. The supporting dataset, `artists.csv`, contains 1,162,095 rows and 5 columns. Each row represents one artist with artist-level information such as followers, popularity, and genre tags.

In [6]:
artists.head()

,id,followers,genres,name,popularity
0,0DheY5irMjBUeLybbCUEZ2,0.0,[],Armid & Amir Zare Pashai feat. Sara Rouzbehani,0
1,0DlhY15l3wsrnlfGio2bjU,5.0,[],ปูนา ภาวิณี,0
2,0DmRESX2JknGPQyO15yxg7,0.0,[],Sadaa,0
3,0DmhnbHjm1qw6NCYPeZNgJ,0.0,[],Tra'gruda,0
4,0Dn11fWM7vHQ3rinvWEl4E,2.0,[],Ioannis Panoutsopoulos,0


In [7]:
artists.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1162095 entries, 0 to 1162094
Data columns (total 5 columns):
 #   Column      Non-Null Count    Dtype  
---  ------      --------------    -----  
 0   id          1162095 non-null  object 
 1   followers   1162084 non-null  float64
 2   genres      1162095 non-null  object 
 3   name        1162092 non-null  object 
 4   popularity  1162095 non-null  int64  
dtypes: float64(1), int64(1), object(3)
memory usage: 44.3+ MB


In [8]:
artists.isna().sum().sort_values(ascending=False)

followers     11
name           3
id             0
genres         0
popularity     0
dtype: int64

In [9]:
tracks.columns

Index(['Unnamed: 0', 'track_id', 'artists', 'album_name', 'track_name',
       'popularity', 'duration_ms', 'release_date', 'explicit', 'danceability',
       'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness',
       'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature',
       'track_genre'],
      dtype='object')

## 2. Data Cleaning


In [10]:
cleaned = tracks.copy()

In [11]:
# Chech misisng values
cleaned.isna().sum().sort_values(ascending=False)

tempo               22114
artists                 1
album_name              1
track_name              1
Unnamed: 0              0
mode                    0
time_signature          0
valence                 0
liveness                0
instrumentalness        0
acousticness            0
speechiness             0
key                     0
loudness                0
track_id                0
energy                  0
danceability            0
explicit                0
release_date            0
duration_ms             0
popularity              0
track_genre             0
dtype: int64

In [12]:
# Check percentage missing
missing_percent = (
    cleaned.isna()
    .mean()
    .sort_values(ascending=False)
    .to_frame("missing_percent")
)

missing_percent.head(20)

,missing_percent
tempo,0.193982
artists,0.000009
album_name,0.000009
track_name,0.000009
Unnamed: 0,0.000000
mode,0.000000
time_signature,0.000000
valence,0.000000
liveness,0.000000
instrumentalness,0.000000


In [13]:
cleaned['release_date']

0               1974
1            1995-04
2               1973
3         2018-08-10
4         2017-02-03
             ...    
113995       2000-04
113996          1973
113997    1992-10-21
113998          1992
113999          1981
Name: release_date, Length: 114000, dtype: object

In [14]:
# Clean release_data
cleaned["release_year"] = cleaned['release_date'].astype(str).str[:4].astype(float) 

In [15]:
cleaned[["release_date", "release_year"]].head()

,release_date,release_year
0,1974,1974.0
1,1995-04,1995.0
2,1973,1973.0
3,2018-08-10,2018.0
4,2017-02-03,2017.0


In [16]:
# Create decade:
cleaned["decade"] = (cleaned["release_year"] // 10 * 10).astype("Int64")

In [17]:
cleaned[["release_date", "release_year", "decade"]].head()

,release_date,release_year,decade
0,1974,1974.0,1970
1,1995-04,1995.0,1990
2,1973,1973.0,1970
3,2018-08-10,2018.0,2010
4,2017-02-03,2017.0,2010


I created `release_year` and `decade` from `release_date` so that I could analyze how music features and popularity change over time.

In [18]:
cleaned["duration_min"] = cleaned["duration_ms"] / 60000

In [19]:
cleaned[["duration_ms", "duration_min"]].head()

,duration_ms,duration_min
0,230666,3.844433
1,149610,2.493500
2,210826,3.513767
3,201933,3.365550
4,198853,3.314217


I converted `duration_ms` into `duration_min` because minutes are easier to interpret than milliseconds.

In [20]:
cleaned["popularity"].describe()

count    114000.000000
mean         33.238535
std          22.305078
min           0.000000
25%          17.000000
50%          35.000000
75%          50.000000
max         100.000000
Name: popularity, dtype: float64

In [21]:
cleaned["popularity"].quantile([0.5, 0.75, 0.8, 0.9, 0.95])

0.50    35.0
0.75    50.0
0.80    54.0
0.90    63.0
0.95    69.0
Name: popularity, dtype: float64

### 2.1 Univariate Analysis

In [22]:
import plotly.express as px

fig = px.histogram(
    cleaned,
    x="popularity",
    nbins=30,
    title="Distribution of Spotify Track Popularity"
)

fig.show()

In [23]:
for cutoff in [50, 55, 60]:
    prop_popular = (cleaned["popularity"] >= cutoff).mean()
    print(cutoff, prop_popular)

50 0.2576052631578947
55 0.1929122807017544
60 0.13001754385964912


In [24]:
popularity_cutoff = 55
cleaned["is_popular"] = cleaned["popularity"] >= popularity_cutoff

I compared popularity cutoffs of 50, 55, and 60. A cutoff of 50 labeled about 25.8% of tracks as popular, while 60 labeled only about 13.0%. I chose 55 because it gives a stricter definition of popularity than the 75th percentile cutoff while still keeping about 19.3% of tracks in the popular class. This makes the classification task more meaningful without making the target too imbalanced.

In [25]:
fig = px.histogram(
    cleaned,
    x="danceability",
    nbins=30,
    title="Distribution of Spotify Track Danceability"
)

fig.show()

This histogram shows the distribution of danceability scores across Spotify tracks. Most tracks have danceability values between 0.4 and 0.8, with the highest concentration around 0.55 to 0.65. This suggests that many songs in the dataset are moderately danceable, while very low-danceability and extremely high-danceability tracks are less common.

In [26]:
fig = px.histogram(
    cleaned,
    x="energy",
    nbins=30,
    title="Distribution of Spotify Track Energy"
)

fig.show()

This histogram shows the distribution of energy scores across Spotify tracks. Most tracks have moderate to high energy values, especially between 0.5 and 1.0, while fewer tracks have very low energy. This suggests that many tracks in the selected genres are relatively active or intense, so energy may be a useful audio feature to include when predicting popularity.

In [27]:
numeric_cols = [
    "popularity",
    "duration_min",
    "danceability",
    "energy",
    "valence",
    "acousticness",
    "instrumentalness",
    "liveness",
    "speechiness",
    "tempo",
    "loudness"
]

cleaned[numeric_cols].describe()

,popularity,duration_min,danceability,energy,valence,acousticness,instrumentalness,liveness,speechiness,tempo,loudness
count,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000,91886.000000,114000.000000
mean,33.238535,3.800486,0.566800,0.641383,0.474068,0.314910,0.156050,0.213553,0.084652,123.119961,-8.258960
std,22.305078,1.788295,0.173542,0.251529,0.259261,0.332523,0.309555,0.190378,0.105732,29.784235,5.029337
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-49.531000
25%,17.000000,2.901100,0.456000,0.472000,0.260000,0.016900,0.000000,0.098000,0.035900,99.999000,-10.013000
50%,35.000000,3.548433,0.580000,0.685000,0.464000,0.169000,0.000042,0.132000,0.048900,122.998000,-7.004000
75%,50.000000,4.358433,0.695000,0.854000,0.683000,0.598000,0.049000,0.273000,0.084500,141.649000,-5.003000
max,100.000000,87.288250,0.985000,1.000000,0.995000,0.996000,1.000000,1.000000,0.965000,222.605000,4.532000


In [28]:
fig = px.box(
    cleaned,
    y="duration_min",
    title="Distribution of Track Duration"
)

fig.show()

In [29]:
cleaned.sort_values("duration_min", ascending=False)[
    ["track_name", "artists", "track_genre", "duration_min", "popularity"]
].head(20)

,track_name,artists,track_genre,duration_min,popularity
73617,Unity (Voyage Mix) Pt. 1,Tale Of Us,minimal-techno,87.288250,35
10984,Crossing Wires 002 - Continuous DJ Mix,Timo Maas,breakbeat,79.817100,11
10935,Crossing Wires 002 - Continuous DJ Mix,Timo Maas,breakbeat,79.817100,11
24348,The Lab 03 - Continuous DJ Mix Part 1,Seth Troxler,detroit-techno,78.838367,8
73840,Amnesia Ibiza Underground 10 DJ Mix,Loco Dice,minimal-techno,76.064950,17
13344,House of Om - Mark Farina - Continuous Mix,Mark Farina,chicago-house,74.125333,11
13245,Live In Tokyo - Continuous Mix,Mark Farina,chicago-house,72.330433,11
13195,Greenhouse Construction,Mark Farina,chicago-house,72.245350,12
27926,"NQ State of Mind, Vol. 1 - Continuous DJ Mix",Lenzman;Dan Stezo,drum-and-bass,70.770100,15
101390,Ocean Waves Sounds,Ocean Sounds,sleep,68.670967,39


In [30]:
cleaned["duration_min"].describe()

count    114000.000000
mean          3.800486
std           1.788295
min           0.000000
25%           2.901100
50%           3.548433
75%           4.358433
max          87.288250
Name: duration_min, dtype: float64

In [31]:
cleaned["duration_min"].quantile([0.5, 0.75, 0.9, 0.95, 0.99])

0.50    3.548433
0.75    4.358433
0.90    5.462883
0.95    6.452785
0.99    8.848461
Name: duration_min, dtype: float64

In [32]:
duration_cutoff = cleaned["duration_min"].quantile(0.99)

cleaned_duration_filtered = cleaned[
    cleaned["duration_min"] <= duration_cutoff
]

In [33]:
cleaned_duration_filtered["duration_min"].describe()

count    112860.000000
mean          3.710253
std           1.278883
min           0.000000
25%           2.895750
50%           3.537117
75%           4.323833
max           8.848433
Name: duration_min, dtype: float64

I checked numeric summaries and possible outliers for important audio features. I did not automatically remove outliers because unusual values may represent real songs, but checking them helps me understand whether any columns contain extreme or suspicious values.

I also checked the distribution of track duration and found that most tracks are between about 3 and 5 minutes long. The median duration is about 3.55 minutes, and 99% of tracks are under about 8.85 minutes. However, the maximum duration is about 87.29 minutes. After inspecting the longest tracks, I found that many were continuous DJ mixes, sleep sounds, or long recordings, so they appear to be real tracks rather than data errors. To reduce the influence of extreme duration values, I used the 99th percentile as a cutoff for some analyses involving duration.

## 2.2 Bivariate Analysis

In [34]:
# Choose at least 5 genres
selected_genres = ["classical", "hip-hop", "country", "electronic", "metal", "pop"]

cleaned = cleaned[cleaned["track_genre"].isin(selected_genres)]

In [35]:
cleaned["track_genre"].value_counts()

track_genre
classical     1000
country       1000
electronic    1000
hip-hop       1000
metal         1000
pop           1000
Name: count, dtype: int64

In [36]:
cleaned["track_genre"].value_counts()

track_genre
classical     1000
country       1000
electronic    1000
hip-hop       1000
metal         1000
pop           1000
Name: count, dtype: int64

In [37]:
cleaned.head()

,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,release_date,explicit,danceability,...,instrumentalness,liveness,valence,tempo,time_signature,track_genre,release_year,decade,duration_min,is_popular
16000,16000,7wrYBASu0OoxoDErd4Edxd,Bombay Jayashri,Rehnaa Hai Terre Dil Mein,Zara Zara,58,298266,2001-12-01,False,0.643,...,0.000002,0.316,0.620,NaN,4,classical,2001.0,2000,4.971100,True
16001,16001,72HdutlIHBZJ7WT1xVAAZT,Shankar;Ehsaan;Loy;Alisha Chinai;Shankar Mahad...,Bunty Aur Babli,Kajra Re,59,482586,2005-04-15,False,0.484,...,0.000000,0.091,0.680,NaN,4,classical,2005.0,2000,8.043100,True
16002,16002,7JGgKHHDgJCJkQCQxyHHdl,Bombay Jayashri;DJ Aftab,Hindi Slowed Reverb Bollywood Lofi,Zara Zara - Lofi,54,219437,1984,False,0.608,...,0.017200,0.448,0.439,140.109,4,classical,1984.0,1980,3.657283,False
16003,16003,3YRj4jmwois2ctPnhwSwFo,Bombay Jayashri,Minnalae,Vaseegara,68,299146,1972,False,0.695,...,0.015800,0.132,0.637,NaN,4,classical,1972.0,1970,4.985767,True
16004,16004,3tp3ij9dtY3CacQgd1OvRf,Bombay Jayashri;Swattrex,Hindi LoFi Vibe,Zara Zara - LoFi Chill,59,387716,1987-06,False,0.583,...,0.010600,0.257,0.241,118.226,4,classical,1987.0,1980,6.461933,True


### Popularity distribution

In [38]:
fig = px.box(
    cleaned,
    x="track_genre",
    y="popularity",
    title="Popularity Distribution by Genre"
)

fig.show()

This box plot compares Spotify popularity scores across selected genres. Pop has the highest median popularity, followed by hip-hop and metal, while classical has the lowest median popularity. This shows that genre may be an important predictor of whether a track is popular, although the wide ranges show that genre alone cannot fully explain popularity.

### Danceability distribution

In [39]:
fig = px.box(
    cleaned,
    x="track_genre",
    y="danceability",
    title="Danceability Distribution by Genre"
)

fig.show()



This box plot shows how danceability varies across the selected genres. Hip-hop has the highest typical danceability, while classical tends to have the lowest. Pop and electronic are also generally more danceable. Which mean that danceability is different across genres, so audio features like this could be useful when comparing or predicting track popularity.

### Energy distribution

In [40]:
fig = px.scatter(
    cleaned,
    x="energy",
    y="popularity",
    color="track_genre",
    opacity=0.5,
    title="Energy vs. Popularity by Genre"
)

fig.show()

This scatter plot looks at the relationship between energy and popularity across the selected genres. Overall, higher energy does not always mean higher popularity, since songs with similar energy levels can have very different popularity scores. Still, the genre colors show some patterns. Pop, hip-hop, and metal tracks appear more often in the higher popularity range, while classical tracks tend to have lower energy and lower popularity. This suggests that energy by itself is not enough to explain popularity, but it could still be useful when combined with genre and other audio features.

## 2.3 Interesting aggregates

In [41]:
genre_summary = (
    cleaned
    .groupby("track_genre")
    .agg(
        num_tracks=("track_id", "count"),
        avg_popularity=("popularity", "mean"),
        median_popularity=("popularity", "median"),
        popular_rate=("is_popular", "mean"),
        avg_danceability=("danceability", "mean"),
        avg_energy=("energy", "mean"),
        avg_valence=("valence", "mean"),
        avg_acousticness=("acousticness", "mean"),
        avg_instrumentalness=("instrumentalness", "mean"),
        avg_tempo=("tempo", "mean")
    )
    .reset_index()
    .sort_values("avg_popularity", ascending=False)
)

genre_summary

,track_genre,num_tracks,avg_popularity,median_popularity,popular_rate,avg_danceability,avg_energy,avg_valence,avg_acousticness,avg_instrumentalness,avg_tempo
5,pop,1000,47.576,66.0,0.644,0.630441,0.606437,0.506223,0.343693,0.009026,122.052689
2,electronic,1000,44.325,48.0,0.310,0.652945,0.694752,0.391839,0.177026,0.248697,123.787936
4,metal,1000,43.705,57.0,0.532,0.464288,0.840273,0.417936,0.037085,0.064603,128.060100
3,hip-hop,1000,37.759,58.0,0.571,0.736154,0.682530,0.551248,0.194175,0.010907,117.118772
1,country,1000,17.028,0.0,0.157,0.555294,0.596805,0.521481,0.321227,0.005585,125.017588
0,classical,1000,13.055,3.0,0.049,0.381923,0.189827,0.381050,0.920049,0.619208,108.432773


In [42]:
genre_summary.round(3)

,track_genre,num_tracks,avg_popularity,median_popularity,popular_rate,avg_danceability,avg_energy,avg_valence,avg_acousticness,avg_instrumentalness,avg_tempo
5,pop,1000,47.576,66.0,0.644,0.630,0.606,0.506,0.344,0.009,122.053
2,electronic,1000,44.325,48.0,0.310,0.653,0.695,0.392,0.177,0.249,123.788
4,metal,1000,43.705,57.0,0.532,0.464,0.840,0.418,0.037,0.065,128.060
3,hip-hop,1000,37.759,58.0,0.571,0.736,0.683,0.551,0.194,0.011,117.119
1,country,1000,17.028,0.0,0.157,0.555,0.597,0.521,0.321,0.006,125.018
0,classical,1000,13.055,3.0,0.049,0.382,0.190,0.381,0.920,0.619,108.433


This table gives a quick summary of how the selected genres differ in popularity and audio features. Pop has the highest popular rate, with about 64% of pop tracks labeled as popular. Hip-hop and metal also have relatively high popular rates, while classical has the lowest. The audio features also show clear genre differences. Hip-hop has the highest average danceability, metal has the highest average energy, and classical has the highest acousticness and instrumentalness but much lower energy. Overall, the table suggests that genre is related to both popularity and the sound of a track, so using genre together with audio features could be helpful for predicting whether a song is popular.

## 3. Assessment of Missingness

In [43]:
missing_summary = (
    cleaned.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("num_missing")
)

missing_summary["percent_missing"] = missing_summary["num_missing"] / len(cleaned)
missing_summary

,num_missing,percent_missing
tempo,1205,0.200833
Unnamed: 0,0,0.000000
track_id,0,0.000000
duration_min,0,0.000000
decade,0,0.000000
release_year,0,0.000000
track_genre,0,0.000000
time_signature,0,0.000000
valence,0,0.000000
liveness,0,0.000000


In [44]:
cleaned["tempo_missing"] = cleaned["tempo"].isna()

In [45]:
cleaned["tempo_missing"].value_counts(normalize=True)

tempo_missing
False    0.799167
True     0.200833
Name: proportion, dtype: float64

In [46]:
observed_ks = stats.ks_2samp(
    cleaned.loc[cleaned["tempo_missing"], "popularity"].dropna(),
    cleaned.loc[~cleaned["tempo_missing"], "popularity"].dropna()
).statistic

observed_ks

np.float64(0.0928076012789948)

### Part A: Test if tempo missingness depends on popularity

In [47]:
n_repetitions = 5000
ks_stats = []

for _ in range(n_repetitions):
    shuffled = cleaned.copy()
    
    shuffled["popularity"] = np.random.permutation(cleaned["popularity"])
    
    groups = shuffled.groupby("tempo_missing")["popularity"]
    
    ks_stat = stats.ks_2samp(
        groups.get_group(True).dropna(),
        groups.get_group(False).dropna()
    ).statistic
    
    ks_stats.append(ks_stat)

ks_stats = np.array(ks_stats)

p_value_popularity = np.mean(ks_stats >= observed_ks)

p_value_popularity

KeyboardInterrupt: 

In [ ]:
fig = px.histogram(
    x=ks_stats,
    nbins=30,
    title="Permutation Test: Tempo Missingness vs. Popularity",
    labels={"x": "KS Statistic"}
)

fig.add_vline(
    x=observed_ks,
    line_dash="dash",
    annotation_text="Observed KS"
)

fig.show()

The permutation test produced a p-value of approximately 0. Since this is below 0.05, I reject the null hypothesis that tempo missingness is independent of popularity. This suggests that whether a track is missing a tempo value depends on the track’s popularity.

### Part B: Test if tempo missingness depends on track_genre

In [49]:
def tvd(data, group_col, cat_col):
    props = (
        data
        .pivot_table(index=cat_col, columns=group_col, aggfunc="size", fill_value=0)
    )
    
    props = props / props.sum()
    
    return 0.5 * np.abs(props[True] - props[False]).sum()

In [50]:
observed_tvd = tvd(cleaned, "tempo_missing", "track_genre")
observed_tvd

np.float64(0.17168644724146437)

In [51]:
n_repetitions = 5000
tvd_stats = []

for _ in range(n_repetitions):
    shuffled = cleaned.copy()
    
    shuffled["track_genre"] = np.random.permutation(cleaned["track_genre"])
    
    stat = tvd(shuffled, "tempo_missing", "track_genre")
    tvd_stats.append(stat)

tvd_stats = np.array(tvd_stats)

p_value_genre = np.mean(tvd_stats >= observed_tvd)

p_value_genre

np.float64(0.0)

In [52]:
fig = px.histogram(
    x=tvd_stats,
    nbins=30,
    title="Permutation Test: Tempo Missingness vs. Genre",
    labels={"x": "TVD"}
)

fig.add_vline(
    x=observed_tvd,
    line_dash="dash",
    annotation_text="Observed TVD"
)

fig.show()

In [49]:
numeric_cols_to_test = [
    "popularity",
    "danceability",
    "energy",
    "valence",
    "acousticness",
    "instrumentalness",
    "liveness",
    "speechiness",
    "duration_min",
    "loudness"
]

results = []

for col in numeric_cols_to_test:
    observed = stats.ks_2samp(
        cleaned.loc[cleaned["tempo_missing"], col].dropna(),
        cleaned.loc[~cleaned["tempo_missing"], col].dropna()
    ).statistic
    
    simulated = []
    
    for _ in range(1000):
        shuffled = cleaned.copy()
        shuffled[col] = np.random.permutation(cleaned[col])
        
        groups = shuffled.groupby("tempo_missing")[col]
        
        stat = stats.ks_2samp(
            groups.get_group(True).dropna(),
            groups.get_group(False).dropna()
        ).statistic
        
        simulated.append(stat)
    
    p_value = np.mean(np.array(simulated) >= observed)
    
    results.append({
        "column": col,
        "observed_ks": observed,
        "p_value": p_value
    })

missingness_results = pd.DataFrame(results).sort_values("p_value")
missingness_results

KeyboardInterrupt: 

In [54]:
def tvd(data, group_col, cat_col):
    props = (
        data
        .pivot_table(index=cat_col, columns=group_col, aggfunc="size", fill_value=0)
    )
    
    props = props / props.sum()
    
    return 0.5 * np.abs(props[True] - props[False]).sum()

In [55]:
categorical_cols_to_test = ["explicit", "mode", "key", "time_signature", "track_genre"]

cat_results = []

for col in categorical_cols_to_test:
    observed = tvd(cleaned, "tempo_missing", col)
    
    simulated = []
    
    for _ in range(1000):
        shuffled = cleaned.copy()
        shuffled[col] = np.random.permutation(cleaned[col])
        
        stat = tvd(shuffled, "tempo_missing", col)
        simulated.append(stat)
    
    p_value = np.mean(np.array(simulated) >= observed)
    
    cat_results.append({
        "column": col,
        "observed_tvd": observed,
        "p_value": p_value
    })

cat_missingness_results = pd.DataFrame(cat_results).sort_values("p_value")
cat_missingness_results

,column,observed_tvd,p_value
3,time_signature,0.063368,0.000
4,track_genre,0.171686,0.000
0,explicit,0.027592,0.007
1,mode,0.038318,0.014
2,key,0.039816,0.587


I tested several numeric columns and found that tempo missingness depends on many of them, including popularity, danceability, energy, valence, acousticness, instrumentalness, speechiness, duration, and loudness. I chose to discuss energy because it had the largest KS statistic, meaning the energy distribution differed the most between tracks with missing tempo and tracks with non-missing tempo.

For categorical columns, I used total variation distance. The missingness of tempo does not appear to depend on key, since the p-value was 0.587. Because this p-value is much larger than 0.05, I do not have enough evidence to say that tempo missingness is related to musical key.

Overall, tempo missingness does not seem to be MCAR, because it depends on observed features such as energy. It is more likely MAR, since the missingness can be partially explained by other observed columns in the dataset. I cannot conclude that it is NMAR from the data alone, because that would require knowing whether missingness depends on the unobserved tempo values themselves.

## 4. Hypothesis Testing 

**Null Hypothesis** : The popularity distribution is the same across all selected genres. Any observed differences in popularity are due to random chance.

**Alternative Hypothesis** : The popularity distribution differs across the selected genres. At least one genre has a different popularity distribution.

**Test Statistic** : Difference between the highest and lowest mean popularity across genres

In [56]:
# Test statistics
def genre_popularity_stat(df):
    genre_means = df.groupby("track_genre")["popularity"].mean()
    return genre_means.max() - genre_means.min()

In [57]:
# Calculate the observed statistics
observed_stat = genre_popularity_stat(cleaned)
observed_stat

np.float64(34.521)

In [58]:
n_repetitions = 5000
simulated_stats = []

for _ in range(n_repetitions):
    shuffled = cleaned.copy()
    shuffled["track_genre"] = np.random.permutation(shuffled["track_genre"])
    
    stat = genre_popularity_stat(shuffled)
    simulated_stats.append(stat)

simulated_stats = np.array(simulated_stats)

In [59]:
p_value = np.mean(simulated_stats >= observed_stat)
p_value

np.float64(0.0)

In [60]:
fig = px.histogram(
    x=simulated_stats,
    nbins=30,
    title="Permutation Test for Popularity Differences Across Genres",
    labels={"x": "Max Mean Popularity - Min Mean Popularity"}
)

fig.add_vline(
    x=observed_stat,
    line_dash="dash",
    annotation_text="Observed Statistic"
)

fig.show()

I used a permutation test to test whether popularity differs across the selected genres. The null hypothesis was that the popularity distribution is the same across all selected genres, while the alternative hypothesis was that at least one genre has a different popularity distribution. My test statistic was the difference between the highest and lowest mean popularity across genres. The p-value was approximately 0, so I reject the null hypothesis at the 0.05 significance level. This provides evidence that genre is associated with track popularity in this dataset.


## 5. Framing a Prediction Problem

#### **Prediction Problem** 

Can we predict whether a Spotify track is popular using its audio features and genre?

#### **Type of Problem**

Binary Classification since we predict categories: popular or not popular

#### **Response Variable**

is_popular 

#### **Features**

danceability

energy

valence

acousticness

instrumentalness

speechiness

liveness

tempo

loudness

duration_min

explicit

track_genre

#### **Evaluation Metric**

I will use F1-score as my main evaluation metric because the classes are imbalanced: only about 19.3% of tracks are labeled as popular. Accuracy may be misleading because a model could get many examples correct by mostly predicting the majority class, “not popular.” F1-score is more appropriate because it balances precision and recall for the popular class.

## Baseline Model 

In [48]:
print(cleaned.dtypes)

Unnamed: 0            int64
track_id             object
artists              object
album_name           object
track_name           object
popularity            int64
duration_ms           int64
release_date         object
explicit               bool
danceability        float64
energy              float64
key                   int64
loudness            float64
mode                  int64
speechiness         float64
acousticness        float64
instrumentalness    float64
liveness            float64
valence             float64
tempo               float64
time_signature        int64
track_genre          object
release_year        float64
decade                Int64
duration_min        float64
is_popular             bool
tempo_missing          bool
dtype: object


In [49]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

baseline_features = ["danceability", "energy", "loudness", "acousticness", "instrumentalness", 
                     "speechiness", "liveness", "valence", "duration_min", "explicit", "track_genre"]

X_base = cleaned[baseline_features]
y = cleaned["is_popular"]

X_train, X_test, y_train, y_test = train_test_split(X_base, y, test_size=0.2, random_state=42)

# 3. Define transformers 
categorical = X_base.select_dtypes(include=['object', 'category']).columns 
numerical = X_base.select_dtypes(include=['number', 'bool']).columns

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numerical),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical)
])

# 4. Build pipeline
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

# 5. Fit and evaluate
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)


print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred, pos_label=True))

Accuracy: 0.8141666666666667
Precision: 0.7867647058823529
Recall: 0.7024070021881839
F1: 0.7421965317919075


In [50]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

baseline_features = ["danceability", "track_genre"]

X_base = cleaned[baseline_features]
y = cleaned["is_popular"]

X_train, X_test, y_train, y_test = train_test_split(X_base, y, test_size=0.2, random_state=42)

# 3. Define transformers 
categorical = X_base.select_dtypes(include=['object', 'category']).columns 
numerical = X_base.select_dtypes(include=['number', 'bool']).columns

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numerical),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical)
])

# 4. Build pipeline
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

# 5. Fit and evaluate
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)


print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred, pos_label=True))


Accuracy: 0.7391666666666666
Precision: 0.6855670103092784
Recall: 0.5820568927789934
F1: 0.6295857988165681


In [51]:
genre_popularity = (cleaned.groupby('track_genre')['is_popular']
                    .mean()
                    .sort_values(ascending=False))
print(genre_popularity.head(10))
print(genre_popularity.tail(10))

track_genre
pop           0.644
hip-hop       0.571
metal         0.532
electronic    0.310
country       0.157
classical     0.049
Name: is_popular, dtype: float64
track_genre
pop           0.644
hip-hop       0.571
metal         0.532
electronic    0.310
country       0.157
classical     0.049
Name: is_popular, dtype: float64


In [52]:
cleaned['num_artists'] = (
    cleaned['artists']
    .str.split(';')
    .str.len()
)
#cleaned["loudness_energy_ratio"] = cleaned['loudness'] / cleaned['energy']
cleaned["tempo_filled"] = cleaned["tempo"].fillna(
    cleaned.groupby("track_genre")["tempo"].transform("median")
)
# Avoid division by zero when creating ratio
cleaned["loudness_energy_ratio"] = cleaned["loudness"] / cleaned["energy"].replace(0, np.nan)


In [53]:
audio_features = ["danceability", "energy", "loudness", "acousticness", 
                  "instrumentalness", "speechiness", "liveness", "valence", 
                  "tempo_filled", "duration_min", "release_year", "num_artists",
                  "loudness_energy_ratio"]

corr_with_target = cleaned[audio_features + ['is_popular']].corr()['is_popular'].sort_values()
print(corr_with_target)

instrumentalness        -0.209486
acousticness            -0.199259
num_artists             -0.038547
liveness                -0.015294
tempo_filled             0.009204
valence                  0.022544
duration_min             0.041544
loudness_energy_ratio    0.069503
speechiness              0.124142
danceability             0.152616
energy                   0.241286
loudness                 0.250726
release_year             0.295150
is_popular               1.000000
Name: is_popular, dtype: float64


In [54]:
corr_plot = corr_with_target.drop('is_popular')

fig2 = px.bar(
    x=corr_plot.values,
    y=corr_plot.index,
    orientation='h',
    title='Feature Correlation with is_popular',
    labels={'x': 'Correlation', 'y': 'Feature'},
    color=corr_plot.values,
    color_continuous_scale='RdBu'
)

fig2.show()

In [55]:
fig2.write_html('/Users/nguyenanh/Spotify_Predictive_Model/assets/feature_correlation.html', include_plotlyjs='cdn')

## 7. Final Model

In [56]:
cleaned.columns

Index(['Unnamed: 0', 'track_id', 'artists', 'album_name', 'track_name',
       'popularity', 'duration_ms', 'release_date', 'explicit', 'danceability',
       'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness',
       'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature',
       'track_genre', 'release_year', 'decade', 'duration_min', 'is_popular',
       'tempo_missing', 'num_artists', 'tempo_filled',
       'loudness_energy_ratio'],
      dtype='object')

In [57]:
from sklearn.model_selection import GridSearchCV

final_features = ["danceability", "acousticness", "energy", "loudness", "instrumentalness",
                  "speechiness", "valence", "duration_min", "explicit", "track_genre", 
                  "loudness_energy_ratio", "tempo_filled", "release_year", "num_artists"]

X_final = cleaned[final_features]
X_train_final, X_test_final, y_train_final, y_test_final = train_test_split(
    X_final, y, test_size=0.2, random_state=42)


# Pipeline (same as before)
categorical = X_final.select_dtypes(include=['object', 'category']).columns.tolist() 
numerical = X_final.select_dtypes(include=['number', 'bool']).columns.tolist()

preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numerical),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical)
])

final_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

# 5. GridSearchCV - fill in the hyperparameters to try!
param_grid = {
    "classifier__n_estimators": [50, 100, 200],
    "classifier__max_depth": [5, 10, 20],
}

grid_search = GridSearchCV(final_pipeline, param_grid, cv=5, scoring="f1")
grid_search.fit(X_train_final, y_train_final)

print("Best params:", grid_search.best_params_)
y_pred_final = grid_search.predict(X_test_final)

print("Accuracy:", accuracy_score(y_test_final, y_pred_final))
print("Precision:", precision_score(y_test_final, y_pred_final))
print("Recall:", recall_score(y_test_final, y_pred_final))
print("F1:", f1_score(y_test_final, y_pred_final, pos_label=True))


Best params: {'classifier__max_depth': 20, 'classifier__n_estimators': 200}
Accuracy: 0.82
Precision: 0.806615776081425
Recall: 0.6936542669584245
F1: 0.7458823529411764


In [58]:
from sklearn.metrics import confusion_matrix
import plotly.figure_factory as ff

# Compute confusion matrix
cm = confusion_matrix(y_test_final, y_pred_final)

# Plot using plotly
labels = ['Not Popular', 'Popular']
fig1 = ff.create_annotated_heatmap(
    cm,
    x=labels,
    y=labels,
    colorscale='Blues',
    showscale=True
)

fig1.update_layout(
    title='Confusion Matrix — Final Model',
    xaxis_title='Predicted',
    yaxis_title='Actual'
)

fig1.show()

In [59]:
fig1.write_html('/Users/nguyenanh/Spotify_Predictive_Model/assets/confusion_matrix.html', include_plotlyjs='cdn')

In [60]:
feature_names = (numerical + categorical)  # or however you defined them
importances = grid_search.best_estimator_.named_steps['classifier'].feature_importances_

# For just the numeric features before one-hot encoding
importance_df = pd.DataFrame({
    'feature': X_train_final.columns,
    'importance': grid_search.best_estimator_.named_steps['classifier'].feature_importances_[:len(X_train_final.columns)]
}).sort_values('importance', ascending=False)

print(importance_df)

                  feature  importance
1            acousticness    0.098025
11           tempo_filled    0.086297
2                  energy    0.081412
9             track_genre    0.078082
7            duration_min    0.074249
0            danceability    0.071554
6                 valence    0.071319
3                loudness    0.070298
5             speechiness    0.070206
10  loudness_energy_ratio    0.066643
4        instrumentalness    0.055309
13            num_artists    0.029442
12           release_year    0.018915
8                explicit    0.013801


In [61]:
from sklearn.inspection import permutation_importance

result = permutation_importance(
    grid_search.best_estimator_, 
    X_test_final, 
    y_test_final, 
    n_repeats=10,
    scoring='f1',
    random_state=42
)

perm_df = pd.DataFrame({
    'feature': X_test_final.columns,
    'importance': result.importances_mean
}).sort_values('importance', ascending=False)

print(perm_df)

                  feature  importance
9             track_genre    0.281631
12           release_year    0.039541
1            acousticness    0.033570
7            duration_min    0.027261
3                loudness    0.022331
0            danceability    0.020754
4        instrumentalness    0.018505
2                  energy    0.018481
8                explicit    0.016196
6                 valence    0.016151
10  loudness_energy_ratio    0.014377
13            num_artists    0.011565
11           tempo_filled    0.004730
5             speechiness    0.004441


In [62]:
fig3 = px.bar(
    perm_df.sort_values('importance'),
    x='importance',
    y='feature',
    orientation='h',
    title='Permutation Importance — Final Model',
    labels={'importance': 'Mean F1 Drop When Shuffled', 'feature': 'Feature'},
    color='importance',
    color_continuous_scale='RdYlGn'

)

fig3.update_xaxes(type='log')
fig3.show()

In [63]:
fig3.write_html('/Users/nguyenanh/Spotify_Predictive_Model/assets/permutation_importance.html', include_plotlyjs='cdn')

## 8. Fairness Analysis

In [65]:
lower_genres = ["classical", "country"]
higher_genres = ["electronic", "hip-hop", "metal", "pop"]
lower_mask = X_test_final['track_genre'].isin(lower_genres)
higher_mask = X_test_final['track_genre'].isin(higher_genres)


f1_lower_genres = f1_score(y_test[lower_mask], y_pred_final[lower_mask], pos_label=True)
f1_higer_genres = f1_score(y_test[higher_mask], y_pred_final[higher_mask], pos_label=True)

observed_diff = f1_lower_genres - f1_higer_genres
print("Observed difference:", observed_diff)

Observed difference: -0.6808461978273299


In [66]:
import numpy as np

n_simulations = 1000
simulated_diffs = []

for _ in range(n_simulations):
    # Shuffle the explicit labels
    shuffled_mask = np.random.permutation(lower_mask)

    f1_shuffled_lower = f1_score(y_test[shuffled_mask], y_pred_final[shuffled_mask], pos_label=True)
    f1_shuffled_higher = f1_score(y_test[~shuffled_mask], y_pred_final[~shuffled_mask], pos_label=True)

    simulated_diffs.append(f1_shuffled_lower - f1_shuffled_higher)

# Calculate p-value
p_value = np.mean(np.array(simulated_diffs) <= observed_diff)
print("P-value:", p_value)

P-value: 0.0


In [67]:
fig4 = px.histogram(
    x=simulated_diffs,
    nbins=30,
    title='Fairness Permutation Test — Lower vs Higher Popularity Genres',
    labels={'x': 'Difference in F1 Score (Lower - Higher Genres)'}
)

fig4.add_vline(
    x=observed_diff,
    line_dash='dash',
    annotation_text=f'Observed: {observed_diff:.3f}'
)

fig4.write_html('/Users/nguyenanh/Spotify_Predictive_Model/assets/fairness_permutation.html', include_plotlyjs='cdn')
fig4.show()

Based on our permutation test, we obtained a p-value of 0.174. Since this is greater than our significance level of 0.05, we fail to reject the null hypothesis. The observed F1 difference of -0.04 between explicit and non-explicit tracks is likely due to randomness.



In [68]:
# Step 1: Calculate observed statistic
explicit_mask = X_test_final['explicit'] == True
non_explicit_mask = X_test_final['explicit'] == False

f1_explicit = f1_score(y_test_final[explicit_mask], y_pred_final[explicit_mask], pos_label=True)
f1_non_explicit = f1_score(y_test_final[non_explicit_mask], y_pred_final[non_explicit_mask], pos_label=True)

observed_diff = f1_explicit - f1_non_explicit
print("Observed difference:", observed_diff)


Observed difference: -0.016505447365804393


In [69]:

# Step 2: Permutation test
n_simulations = 1000
simulated_diffs = []

for _ in range(n_simulations):
    shuffled_explicit = np.random.permutation(explicit_mask)
    
    f1_shuffled_explicit = f1_score(y_test_final[shuffled_explicit], y_pred_final[shuffled_explicit], pos_label=True)
    f1_shuffled_non_explicit = f1_score(y_test_final[~shuffled_explicit], y_pred_final[~shuffled_explicit], pos_label=True)
    
    simulated_diffs.append(f1_shuffled_explicit - f1_shuffled_non_explicit)

# Step 3: Calculate p-value
p_value = np.mean(np.array(simulated_diffs) <= observed_diff)
print("P-value:", p_value)


P-value: 0.395


In [70]:

# Step 4: Visualization
fig5 = px.histogram(
    x=simulated_diffs,
    nbins=30,
    title='Fairness Permutation Test — Explicit vs Non-Explicit Tracks',
    labels={'x': 'Difference in F1 Score (Explicit - Non-Explicit)'}
)

fig5.add_vline(
    x=observed_diff,
    line_dash='dash',
    annotation_text=f'Observed: {observed_diff:.3f}'
)

fig5.write_html('/Users/nguyenanh/Spotify_Predictive_Model/assets/fairness_explicit.html', include_plotlyjs='cdn')
fig5.show()